# SmartBite PARSeq Date-Synth Full Colab Workflow

This notebook fine-tunes PARSeq on the existing PP-OCRv5 recognition dataset layout:

1. Mount Drive
2. Unzip `ppocrv5_date_synth_dataset.zip`
3. Validate `train_images/`, `val_images/`, `train_label.txt`, `val_label.txt`
4. Install PARSeq training dependencies
5. Convert PP-OCR recognition labels to PARSeq LMDB datasets
6. Train PARSeq with `subprocess.Popen` live logs
7. Export and save artifacts back to Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


In [ ]:
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path


def run_live(cmd, env=None, cwd=None):
    print('>>', shlex.join([str(c) for c in cmd]))
    proc = subprocess.Popen(
        [str(c) for c in cmd],
        env=env,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)


## Config
Update only these paths/values if needed.


In [ ]:
# Zip in Drive (source of truth)
DATASET_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/ppocrv5_date_synth_dataset.zip')

# Optional Drive-hosted PARSeq repo archive. If missing, the notebook clones from GitHub.
PARSEQ_ZIP_DRIVE = Path('/content/drive/My Drive/sb-colab/parseq.zip')

# Fresh local working directories
BASE_UNZIP_DIR = Path('/content/dataset')
PARSEQ_DIR = Path('/content/parseq')
PARSEQ_DATA_ROOT = Path('/content/parseq_data')

# Training config
RUN_NAME = 'smartbite_parseq_date_synth_full'
OUTPUT_DIR = Path('/content/output') / RUN_NAME
FINAL_MODEL_DRIVE_DIR = Path('/content/drive/My Drive/sb-colab/models') / RUN_NAME
EPOCHS = 30
BATCH_SIZE = 128
IMG_SIZE = [32, 128]
NUM_WORKERS = 2
DEVICE = 'gpu'  # set 'cpu' if no GPU


In [ ]:
assert DATASET_ZIP_DRIVE.exists(), f'Missing dataset zip: {DATASET_ZIP_DRIVE}'

if BASE_UNZIP_DIR.exists():
    shutil.rmtree(BASE_UNZIP_DIR)
BASE_UNZIP_DIR.mkdir(parents=True, exist_ok=True)

run_live(['unzip', '-q', '-o', DATASET_ZIP_DRIVE, '-d', BASE_UNZIP_DIR])


def find_dataset_root(base: Path) -> Path:
    if (base / 'train_label.txt').exists() and (base / 'val_label.txt').exists():
        return base
    for candidate in sorted(base.rglob('*')):
        if not candidate.is_dir():
            continue
        if (candidate / 'train_label.txt').exists() and (candidate / 'val_label.txt').exists():
            return candidate
    raise FileNotFoundError('Could not find dataset root containing train_label.txt and val_label.txt')


DATASET_ROOT = find_dataset_root(BASE_UNZIP_DIR)
print('DATASET_ROOT =', DATASET_ROOT)

for split in ('train', 'val'):
    label_file = DATASET_ROOT / f'{split}_label.txt'
    lines = [line for line in label_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    missing = 0
    bad = 0
    for line in lines:
        if '\t' not in line:
            bad += 1
            continue
        rel_path, text = line.split('\t', 1)
        if not text.strip():
            bad += 1
        if not (DATASET_ROOT / rel_path).exists():
            missing += 1
    print(f'{split}: labels={len(lines)} missing_files={missing} bad_lines={bad}')
    assert missing == 0, f'{split} has missing image files'
    assert bad == 0, f'{split} has malformed or empty labels'

assert sum(1 for _ in open(DATASET_ROOT / 'train_label.txt', 'r', encoding='utf-8')) == 115200
assert sum(1 for _ in open(DATASET_ROOT / 'val_label.txt', 'r', encoding='utf-8')) == 12800


## Install PARSeq
This mirrors the Drive-first style of the PP-OCRv5 notebook: if `parseq.zip` exists in Drive it is used; otherwise the official repo is cloned.


In [ ]:
if PARSEQ_DIR.exists():
    shutil.rmtree(PARSEQ_DIR)

if PARSEQ_ZIP_DRIVE.exists():
    run_live(['unzip', '-q', '-o', PARSEQ_ZIP_DRIVE, '-d', '/content'])
    candidates = [p for p in Path('/content').glob('*') if p.is_dir() and (p / 'train.py').exists()]
    assert candidates, 'parseq.zip did not extract to a directory containing train.py'
    extracted = sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]
    if extracted != PARSEQ_DIR:
        shutil.move(str(extracted), str(PARSEQ_DIR))
else:
    run_live(['git', 'clone', '--depth', '1', 'https://github.com/baudm/parseq.git', PARSEQ_DIR])

run_live([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'])
# Install PARSeq package plus explicit training dependencies.
# The repo extras can be incomplete on Colab, so keep the core stack explicit.
run_live([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=PARSEQ_DIR)
run_live([
    sys.executable, '-m', 'pip', 'install', '-q',
    'numpy<2',
    'pytorch-lightning',
    'torchmetrics',
    'hydra-core',
    'timm',
    'nltk',
    'lmdb',
    'imgaug',
    'Pillow',
    'opencv-python-headless',
])

if os.system('nvidia-smi > /dev/null 2>&1') == 0:
    run_live(['nvidia-smi'])
else:
    DEVICE = 'cpu'
    print('No GPU detected; DEVICE set to cpu')


## Convert PP-OCR Labels To PARSeq LMDB
PARSeq training expects LMDB datasets. The source PP-OCR labels are already recognition pairs: `relative/image/path<TAB>date text`.


In [ ]:
import lmdb


def write_lmdb(label_file: Path, output_dir: Path) -> int:
    if output_dir.exists():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    lines = [line for line in label_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    env = lmdb.open(str(output_dir), map_size=1024 * 1024 * 1024)
    count = 0
    with env.begin(write=True) as txn:
        for line in lines:
            rel_path, text = line.split('\t', 1)
            img_path = DATASET_ROOT / rel_path
            if not img_path.exists():
                raise FileNotFoundError(img_path)
            count += 1
            txn.put(f'image-{count:09d}'.encode(), img_path.read_bytes())
            txn.put(f'label-{count:09d}'.encode(), text.strip().encode('utf-8'))
        txn.put('num-samples'.encode(), str(count).encode())
    env.sync()
    env.close()
    return count


if PARSEQ_DATA_ROOT.exists():
    shutil.rmtree(PARSEQ_DATA_ROOT)

# PARSeq config `dataset=real` looks under train/real/* and val/*.
train_lmdb = PARSEQ_DATA_ROOT / 'train' / 'real' / 'smartbite_date_synth_full'
val_lmdb = PARSEQ_DATA_ROOT / 'val' / 'smartbite_date_synth_full'
train_n = write_lmdb(DATASET_ROOT / 'train_label.txt', train_lmdb)
val_n = write_lmdb(DATASET_ROOT / 'val_label.txt', val_lmdb)

print('train_lmdb =', train_lmdb, 'samples =', train_n)
print('val_lmdb   =', val_lmdb, 'samples =', val_n)
assert train_n == 115200
assert val_n == 12800


## Train PARSeq (Popen)


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Use direct shell execution for PARSeq training.
# PyTorch Lightning progress bars do not stream reliably through subprocess.PIPE,
# but Colab shell execution behaves much closer to an interactive terminal.
train_py = PARSEQ_DIR / 'train.py'
cmd_text = f'''
PYTHONUNBUFFERED=1 HYDRA_FULL_ERROR=1 python -u {train_py} \
  +experiment=parseq \
  pretrained=parseq \
  dataset=real \
  data.root_dir={PARSEQ_DATA_ROOT} \
  model.batch_size={BATCH_SIZE} \
  "model.img_size={IMG_SIZE}" \
  model.max_label_length=25 \
  trainer.max_epochs={EPOCHS} \
  trainer.accelerator={DEVICE} \
  trainer.devices=1 \
  trainer.val_check_interval=1.0 \
  data.num_workers={NUM_WORKERS} \
  hydra.run.dir={OUTPUT_DIR} \
  +trainer.enable_progress_bar=True \
  +trainer.log_every_n_steps=10 \
  +trainer.num_sanity_val_steps=0 \
  ++data.augment=False
'''.strip()

print('>>', cmd_text)
%cd {PARSEQ_DIR}
!{cmd_text}
print('Training command finished. Output dir:', OUTPUT_DIR)


## Inspect And Export Best Checkpoint


In [ ]:
import torch

ckpts = sorted(OUTPUT_DIR.rglob('*.ckpt'), key=lambda p: p.stat().st_mtime, reverse=True)
print('checkpoints:', [str(p) for p in ckpts[:10]])
assert ckpts, f'No .ckpt files found under {OUTPUT_DIR}'

best_ckpt = next((p for p in ckpts if 'last' not in p.name.lower()), ckpts[0])
print('Selected checkpoint:', best_ckpt)

EXPORT_DIR = OUTPUT_DIR / 'smartbite_export'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
payload = torch.load(best_ckpt, map_location='cpu')
state_dict = payload.get('state_dict', payload)
torch.save(state_dict, EXPORT_DIR / 'pytorch_model.bin')
(EXPORT_DIR / 'source_checkpoint.txt').write_text(str(best_ckpt), encoding='utf-8')

print('Exported:', EXPORT_DIR / 'pytorch_model.bin')
print('Export size MB:', round((EXPORT_DIR / 'pytorch_model.bin').stat().st_size / 1024 / 1024, 2))


## Smoke-Test Inference On Validation Samples


In [ ]:
sample_lines = [line for line in (DATASET_ROOT / 'val_label.txt').read_text(encoding='utf-8').splitlines() if line.strip()][:8]
sample_paths = [DATASET_ROOT / line.split('\t', 1)[0] for line in sample_lines]
print('Ground truth:')
for line in sample_lines:
    print(line)

# PARSeq read.py accepts a checkpoint and one or more image paths.
run_live([sys.executable, str(PARSEQ_DIR / 'read.py'), str(best_ckpt), *sample_paths], cwd=PARSEQ_DIR)


## Backup Artifacts To Drive


In [ ]:
assert OUTPUT_DIR.exists(), f'Missing output dir: {OUTPUT_DIR}'
if FINAL_MODEL_DRIVE_DIR.exists():
    shutil.rmtree(FINAL_MODEL_DRIVE_DIR)
FINAL_MODEL_DRIVE_DIR.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(OUTPUT_DIR, FINAL_MODEL_DRIVE_DIR)
print('Saved PARSeq artifacts to:', FINAL_MODEL_DRIVE_DIR)
print('SmartBite export:', FINAL_MODEL_DRIVE_DIR / 'smartbite_export' / 'pytorch_model.bin')


## Notes
- This dataset is appropriate for PARSeq because it has cropped recognition images and exact transcriptions.
- It is not a text-detection dataset; no text boxes are needed for PARSeq.
- The exported `pytorch_model.bin` is intended for SmartBite's eager PARSeq loader path.
- If training overfits quickly, reduce `EPOCHS` or add more date-recognition crops.
